Import libraries. Will probably have to install prophet first.

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
from prophet import Prophet
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_percentage_error

sns.set_style("whitegrid")

Load Dataset

In [ ]:
file_path = "future-gc00-daily-prices.csv"
gold_ts = pd.read_csv(file_path)

Clean data, and sort by date. In order to use Prophet, the columns need to be labelled ds (for datestamp) and y (for target). Obviously, we will have to change the code below a bit for this.

In [ ]:
for col in ["Open", "High", "Low", "Close"]:
    gold_ts[col] = gold_ts[col].str.replace(",", "").astype(float)
gold_ts["Date"] = pd.to_datetime(gold_ts["Date"], format="%m/%d/%Y")

gold_ts = gold_ts.sort_values("Date")

#In order to add sunshine_duration as a regressor, we need to add it to the dataframe that we're applying Prophet to.
all_weather_df = pd.read_csv('spring-2026-solar-panel-degradation-team-2/notebooks/eda/AM/weather_CSVs/2105_weather.csv')
all_weather_df['date'] = pd.to_datetime(all_weather_df['date'])
sunshine_df = system_df[['date','sunshine_duration']]

system_df = pd.merge(gold_ts, sunshine_df, on = 'date', how = 'inner')
system_df = system_df[["Date", "Close"]].rename(columns={"Date": "ds", "Close": "y"})

Make train/test split. This would also need to be adjusted for our modelling.

In [ ]:
mapes = []
forecasts = []

for i, date in enumerate(pd.date_range('2021–01–01', '2021–12–31', freq='M')):
    #Make the rolling window split
    train = system_df.loc[date - pd.offsets.Day(14):date]
    test = system_df.loc[date:date + pd.offsets.Day(1)] #<-- need to adjust to avoid overlap at boundary
    
    #Fit Prophet model. Here, daily and yearly seasonal components are enabled.
    model = Prophet(daily_seasonality=True, yearly_seasonality=True)
    model.add_regressor('sunshine_duration') #<-- gold_ts must have "sunshine_duration" as a column
    model.fit(train) 

    #Forecast future values. Number of days predicted is represented by periods argument.
    #The forecast will return columns yhat (predicted value), yhat_lower, yhat_upper.
    future = model.make_future_dataframe(periods=1, freq='D', include_history=True)
    
    #In order for Prophet to make a forecast, we need to have sunshine_duration data for the forecast window.
    #We will call a sunshine_duration() function that adds this data to the future dataframe.
    future['sunshine_duration'] = future['ds'].apply(sunshine_duration)
    forecasts[i] = model.predict(future)

    #Compare forecasted values to actual values, and compute MAPE.
    merged = pd.merge(test, (forecasts[i])[["ds", "yhat"]], on="ds", how="inner")

    if not merged.empty:
        mapes[i] = mean_absolute_percentage_error(merged["y"], merged["yhat"]) * 100
        

Plot the forecasts.

In [ ]:
for i in forecasts:
    fig1 = model.plot(forecasts[i])
    plt.title("Gold Futures (GC00) - Close Price Forecast")
    plt.xlabel("Date")
    plt.ylabel("Close Price")
    plt.show()

Plot seasonal components (trend, weekly, yearly, daily)

In [ ]:
for i in forecasts:
    fig2 = model.plot_components(forecasts[i])
    plt.show()

Review forecast samples.

In [ ]:
for i in forecasts:
    print(forecasts[i][["ds", "yhat", "yhat_lower", "yhat_upper"]].tail(10))

### Copied code that got repurposed.

Fit Prophet model. Here, daily and yearly seasonal components are enabled.

In [ ]:
model = Prophet(daily_seasonality=True, yearly_seasonality=True)
model.fit(train)

Forecast future values. Number of days predicted is equal to the size of the test set. The forecast will return columns yhat (predicted value), yhat_lower, yhat_upper.

In [ ]:
future = model.make_future_dataframe(periods=test_size)
forecast = model.predict(future)

Compare forecasted values to actual values, and compute MAPE.

In [ ]:
merged = pd.merge(test, forecast[["ds", "yhat"]], on="ds", how="inner")

if merged.empty:
    print("No overlapping dates between forecast and test set.")
else:
    mape = mean_absolute_percentage_error(merged["y"], merged["yhat"]) * 100
    print(f"Mean Absolute Percentage Error (MAPE): {mape:.2f}%")